In [ ]:
# ── INSTALAÇÃO AUTOMÁTICA DE DEPENDÊNCIAS ────────────────────────────────────
import subprocess, sys, importlib

PACOTES = {
    'pandas': 'pandas>=2.0', 'numpy': 'numpy>=1.26',
    'matplotlib': 'matplotlib>=3.8', 'pyarrow': 'pyarrow>=15.0',
    'scipy': 'scipy>=1.11', 'statsmodels': 'statsmodels>=0.14',
}
instalou = False
for modulo, pacote in PACOTES.items():
    try:
        importlib.import_module(modulo)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pacote], check=False)
        instalou = True

if instalou:
    try:
        import IPython; IPython.Application.instance().kernel.do_shutdown(restart=True)
    except Exception:
        print("Reinicie o kernel manualmente e rode novamente.")
else:
    print("✓ Dependências OK. Pode continuar.")

# Aula 3 — Análise Exploratória dos Dados
**Intensivão Quant AI — ImpactUFSCar**

Antes de construir qualquer sinal, precisamos entender os dados. Hoje: o vocabulário estatístico que aparece em todo relatório quant aplicado diretamente nos dados do IBOVESPA.

In [ ]:
# ── AMBIENTE ──────────────────────────────────────────────────────────────────
import os, subprocess
try:
    import google.colab; IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    REPO_DIR = '/content/intensivao-quant-ai'
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git','clone','https://github.com/fmaldonadoo/intensivao-quant-ai.git', REPO_DIR])
    DADOS_DIR = '/content/drive/MyDrive/intensivao_quant/dados'
else:
    _p = os.path.abspath(os.getcwd()); _root = None
    while _p != os.path.dirname(_p):
        if os.path.exists(os.path.join(_p, '.git')): _root = _p; break
        _p = os.path.dirname(_p)
    if _root is None: _root = os.path.dirname(os.path.abspath('__file__'))
    DADOS_DIR = os.path.join(_root, 'dados')

os.makedirs(DADOS_DIR, exist_ok=True)
print(f"Pasta de dados: {DADOS_DIR}")

In [ ]:
# ── VERIFICAÇÃO ───────────────────────────────────────────────────────────────
_necessarios = ['precos_ibov.parquet', 'retornos_diarios.parquet', 'retornos_mensais.parquet']
_faltando = [f for f in _necessarios if not os.path.exists(os.path.join(DADOS_DIR, f))]
if _faltando:
    print("⚠  Faltando:", _faltando, "→ rode aula-02-dados primeiro")
else:
    print("✓ Dados encontrados.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

precos      = pd.read_parquet(os.path.join(DADOS_DIR, 'precos_ibov.parquet'))
retornos    = pd.read_parquet(os.path.join(DADOS_DIR, 'retornos_diarios.parquet'))
ret_mensais = pd.read_parquet(os.path.join(DADOS_DIR, 'retornos_mensais.parquet'))

print(f"Preços: {precos.shape}  |  Diários: {retornos.shape}  |  Mensais: {ret_mensais.shape}")

## 1. Esperança, Variância e Risco × Retorno

**Esperança (μ):** retorno médio. **Volatilidade (σ):** desvio padrão — nossa proxy de risco.

Anualização: `μ_anual = μ_diária × 252` e `σ_anual = σ_diária × √252`.

O scatter abaixo mostra o trade-off fundamental: queremos o canto superior esquerdo (alto retorno, baixo risco). O **Sharpe Ratio** (Aula 5) formaliza isso.

In [ ]:
resumo = pd.DataFrame({
    'retorno_anual': retornos.mean() * 252,
    'vol_anual':     retornos.std()  * np.sqrt(252)
}).dropna().sort_values('retorno_anual', ascending=False)

print("Top 5:");  print(resumo.head().to_string(float_format='{:.2%}'.format))
print("\nBottom 5:"); print(resumo.tail().to_string(float_format='{:.2%}'.format))

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(resumo['vol_anual'], resumo['retorno_anual'], alpha=0.6, s=50)
for t in ['WEGE3.SA','PETR4.SA','VALE3.SA','MGLU3.SA','BBAS3.SA']:
    if t in resumo.index:
        ax.annotate(t.replace('.SA',''), (resumo.loc[t,'vol_anual'], resumo.loc[t,'retorno_anual']),
                    textcoords='offset points', xytext=(6,3), fontsize=8)
ax.axhline(0, color='red', linewidth=0.8, linestyle='--')
ax.set_xlabel('Volatilidade anual'); ax.set_ylabel('Retorno anual médio')
ax.set_title('Risco × Retorno — IBOVESPA (2010–2024)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.0%}'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.0%}'))
plt.tight_layout(); plt.show()

## 2. Fat Tails — Por Que a Normal Mente

- **Skewness (assimetria):** Normal = 0. Retornos têm skewness negativa — quedas são mais rápidas e extremas que altas.
- **Kurtosis (curtose):** Normal = 3. Retornos têm curtose > 3 (**fat tails**) — eventos extremos ocorrem muito mais do que a Normal prevê.

| Evento | Frequência pela Normal | Frequência real |
|---|---|---|
| \|r\| > 3σ | 1 vez a cada 370 dias | Várias vezes por ano |
| \|r\| > 5σ | 1 vez a cada 4.700 anos | Em quase toda grande crise |
| \|r\| > 7σ | Praticamente impossível | COVID mar/2020, Joesley Day 2017 |

**Conclusão:** use **Max Drawdown** e **Sortino Ratio** — não confie só no Sharpe.

In [ ]:
petr4_ret = retornos['PETR4.SA'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.linspace(petr4_ret.min(), petr4_ret.max(), 300)
axes[0].hist(petr4_ret, bins=100, density=True, alpha=0.6, color='steelblue', label='retornos reais')
axes[0].plot(x, stats.norm.pdf(x, petr4_ret.mean(), petr4_ret.std()), 'r-', lw=2, label='normal ajustada')
axes[0].set_title('Histograma — PETR4'); axes[0].set_xlim(-0.15, 0.15); axes[0].legend()

stats.probplot(petr4_ret, dist='norm', plot=axes[1])
axes[1].set_title('QQ-Plot — fat tails nas extremidades')
axes[1].get_lines()[1].set_color('red')

plt.tight_layout(); plt.show()

print(f"Curtose (excess): {petr4_ret.kurtosis():.2f}  (normal = 0)")
print(f"Assimetria:       {petr4_ret.skew():.2f}   (normal = 0)")
sigma = petr4_ret.std()
print("\nEventos extremos:")
for n in [3, 4, 5]:
    obs = (petr4_ret.abs() > n*sigma).sum()
    esp = len(petr4_ret) * 2 * stats.norm.sf(n)
    print(f"  |r| > {n}σ: observado = {obs:3d}  |  esperado pela normal = {esp:.1f}")

## 3. Estacionariedade

- **Preços → não estacionários:** têm tendência crescente. Regressões entre preços não estacionários encontram relações que não existem — **regressão espúria**.
- **Retornos → estacionários:** oscilam em torno de uma média próxima de zero.

**Regra:** sempre modele retornos, nunca preços.

Teste ADF: H₀ = não estacionária. p-value < 0.05 → série é estacionária ✓

In [ ]:
from statsmodels.tsa.stattools import adfuller

petr4_preco = precos['PETR4.SA'].dropna()

fig, axes = plt.subplots(2, 1, figsize=(13, 6))
petr4_preco.plot(ax=axes[0], color='steelblue')
axes[0].set_title('PETR4 — Preço (não estacionário)'); axes[0].set_ylabel('Preço (R$)')
petr4_ret.plot(ax=axes[1], color='steelblue', alpha=0.7)
axes[1].axhline(petr4_ret.mean(), color='red', lw=1.2, ls='--', label=f'média = {petr4_ret.mean():.4f}')
axes[1].set_title('PETR4 — Retorno diário (estacionário)'); axes[1].legend()
plt.tight_layout(); plt.show()

for nome, serie in [('Preço', petr4_preco), ('Retorno', petr4_ret)]:
    p = adfuller(serie.dropna())[1]
    print(f"ADF {nome}: p-value = {p:.4f}  → {'Estacionária ✓' if p < 0.05 else 'Não estacionária ✗'}")

## 4. Autocorrelação: O Mercado Tem Memória?

- **≈ 0 no diário:** retornos de hoje não preveem amanhã — mercados eficientes na forma fraca.
- **Positiva no mensal (momentum):** retornos positivos nos últimos 12 meses tendem a persistir. É o fenômeno que nossa estratégia explora.

Barras fora da área azul (intervalo de confiança) são estatisticamente significativas.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_acf(petr4_ret.dropna(), lags=30, ax=axes[0], alpha=0.05)
axes[0].set_title('ACF — PETR4 Retornos DIÁRIOS'); axes[0].set_xlabel('Lag (dias)')
plot_acf(ret_mensais['PETR4.SA'].dropna(), lags=24, ax=axes[1], alpha=0.05)
axes[1].set_title('ACF — PETR4 Retornos MENSAIS'); axes[1].set_xlabel('Lag (meses)')
plt.tight_layout(); plt.show()

print(f"Autocorr 1 dia:    {petr4_ret.autocorr(lag=1):.4f}")
print(f"Autocorr 1 mês:    {ret_mensais['PETR4.SA'].autocorr(lag=1):.4f}")
print(f"Autocorr 12 meses: {ret_mensais['PETR4.SA'].autocorr(lag=12):.4f}")

## 5. Filtros: Cobertura e Liquidez

**Cobertura:** removemos ações com menos de 80% de dias com dado válido (IPOs recentes, suspensões longas).

**Liquidez:** removemos os 10% menos líquidos usando % de dias com retorno ≠ 0 como proxy (ações ilíquidas ficam dias sem negociação, o spread elimina o retorno da estratégia).

In [ ]:
# Filtro de cobertura
cobertura = retornos.notna().mean()
tickers_validos   = cobertura[cobertura >= 0.80].index.tolist()
tickers_removidos = cobertura[cobertura < 0.80].index.tolist()
print(f"Cobertura < 80% → removidos: {len(tickers_removidos)}")
print(f"  {[t.replace('.SA','') for t in tickers_removidos]}")

# Filtro de liquidez
liquidez = (retornos[tickers_validos] != 0).mean()
tickers_finais = liquidez[liquidez >= liquidez.quantile(0.10)].index.tolist()
print(f"\nApós filtro de liquidez (p10): {len(tickers_finais)} ações")

## 6. Salvando o Dataset Limpo

In [ ]:
precos[tickers_finais].to_parquet(os.path.join(DADOS_DIR, 'precos_limpo.parquet'))
retornos[tickers_finais].to_parquet(os.path.join(DADOS_DIR, 'retornos_diarios_limpo.parquet'))
ret_mensais[tickers_finais].to_parquet(os.path.join(DADOS_DIR, 'retornos_mensais_limpo.parquet'))
pd.Series(tickers_finais, name='ticker').to_csv(os.path.join(DADOS_DIR, 'tickers_finais.csv'), index=False)

print(f"Dataset limpo: {len(tickers_finais)} ações × {retornos.shape[0]} dias")
print("Arquivos salvos: precos_limpo | retornos_diarios_limpo | retornos_mensais_limpo | tickers_finais")

## Resumo

| Conceito | Regra prática |
|---|---|
| Esperança / volatilidade | Base do Sharpe Ratio (Aula 5) |
| Fat tails | Drawdowns reais maiores que modelos normais preveem |
| Estacionariedade | Sempre modele retornos, nunca preços |
| Autocorrelação | Momentum é fenômeno mensal — justifica janela 12 meses |
| Filtros | 4 arquivos limpos salvos — matéria-prima das próximas aulas |

---
*ImpactUFSCar — Diretoria de Quant*